In [0]:
%python
from pyspark.sql.functions import (
    col, to_date, upper, trim, round, when, lit, current_timestamp
)
from pyspark.sql.types import IntegerType, DoubleType

# Unity Catalog : toujours utiliser la notation 3 niveaux
df = spark.table("hive_metastore.bronze.ventes_raw")

# Étape 1 — Typage
df_typed = (
    df
    .withColumn("order_id",      col("order_id").cast(IntegerType()))
    .withColumn("date_commande", to_date(col("date_commande"), "yyyy-MM-dd"))
    .withColumn("qte",           col("qte").cast(IntegerType()))
    .withColumn("prix_unitaire", col("prix_unitaire").cast(DoubleType()))
    .withColumn("remise",        col("remise").cast(DoubleType()))
)

# Étape 2 — Nettoyage textuel
df_clean = (
    df_typed
    .withColumn("nom_client", trim(col("nom_client")))
    .withColumn("produit",    trim(col("produit")))
    .withColumn("categorie",  upper(trim(col("categorie"))))
    .withColumn("region",     trim(col("region")))
)

# Étape 3 — Enrichissement
df_enriched = (
    df_clean
    .withColumn("montant_brut", round(col("qte") * col("prix_unitaire"), 2))
    .withColumn("montant_net",  round(col("montant_brut") * (1 - col("remise")), 2))
    .withColumn("segment_prix",
        when(col("prix_unitaire") < 50,  lit("Petit prix"))
        .when(col("prix_unitaire") < 500, lit("Milieu de gamme"))
        .otherwise(lit("Premium"))
    )
    .withColumn("_silver_ts", current_timestamp())
)

# Étape 4 — Qualité
df_valid = (
    df_enriched
    .dropDuplicates(["order_id"])
    .filter(col("order_id").isNotNull())
    .filter(col("qte") > 0)
    .filter(col("prix_unitaire") > 0)
)

# Étape 5 — Écriture Silver
(
    df_valid.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", "abfss://silver@adlscommercialprod.dfs.core.windows.net/ventes_clean")
    .saveAsTable("hive_metastore.silver.ventes_clean")
)

print(f"✅ Silver : {df_valid.count()} lignes valides")
df_valid.select("order_id","nom_client","montant_net","segment_prix").show(5)

✅ Silver : 8 lignes valides
+--------+------------+-----------+---------------+
|order_id|  nom_client|montant_net|   segment_prix|
+--------+------------+-----------+---------------+
|       1|   Dupont SA|     2160.0|        Premium|
|       6| Girard Tech|     719.92|Milieu de gamme|
|       3|   Dupont SA|     664.05|        Premium|
|       5| Martin SARL|     299.98|Milieu de gamme|
|       4|Leblanc Fils|     3060.0|        Premium|
+--------+------------+-----------+---------------+
only showing top 5 rows



In [0]:
SELECT categorie, COUNT(*), SUM(montant_net) FROM hive_metastore.silver.ventes_clean GROUP BY categorie;

categorie,count(1),sum(montant_net)
INFORMATIQUE,4,7142.25
PERIPHERIQUES,3,1469.85
STOCKAGE,1,756.2
